[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/062.%20The%20Picker%20Routing%20Problem/P62-Tier-3_executed.ipynb)

---


# 62. The Picker Routing Problem

## Tier 3: Firefly Algorithm Implementation

This notebook applies a population-based Firefly Algorithm to a picker-routing instance. Each candidate solution is a complete ordering of the items to collect. The algorithm repeatedly uses route quality, similarity between route orderings, attraction, and random changes to search for a short tour.

### Goals, inputs, and outputs

- **Goal:** Find a short route that starts at the depot, visits every listed item once, and returns to the depot.
- **Inputs:** Item identifiers and their two-dimensional coordinates; a depot coordinate; and algorithm parameters such as population size and attraction strength.
- **Outputs:** The best route found by the search, its distance, iteration history, population-diversity history, and computation time.
- **Method:** A discrete, permutation-based adaptation of the Firefly Algorithm. It is a stochastic search method, so it does not certify that the route found is globally optimal.

### Model assumptions and scope

Coordinates are treated as points in a two-dimensional Euclidean plane, and the distance between consecutive stops is their straight-line distance. Thus, the model does not represent aisle networks, one-way passages, blocked areas, turns, congestion, item-picking time, or capacity limits. Distances are reported in coordinate distance units because no physical unit is specified. The route is a closed tour: it begins and ends at the depot.

The Firefly Algorithm was originally described for continuous search spaces. Here, candidate solutions are permutations, so attraction is implemented through order-based route crossover, and random movement is implemented through permutation mutations. The algorithm's route-similarity measure is Hamming distance between item positions; it is distinct from the physical travel distance used to evaluate a route.

### Search procedure

1. **Initialize:** Create a population of randomly ordered routes.
2. **Evaluate:** Compute each route's closed-tour distance and brightness.
3. **Attract:** Move the dimmer member of each pair toward the brighter member with a probability that decreases with route similarity distance.
4. **Explore:** Apply a random permutation change with probability `alpha`.
5. **Track:** Preserve the best route found and record the best distance and population diversity.
6. **Stop:** Finish at the iteration limit or when the best distance changes by less than the convergence threshold over the specified comparison window.

### Example instance

The example uses 10 item locations, a population of 30 fireflies, and a maximum of 1,000 iterations. The item coordinates and algorithm settings are defined in the code cells below. Since the search is stochastic, its measured improvement and stopping iteration are outputs of a run, not guaranteed target values.

### Reading the results

The best-distance history shows how the best route found changes during the search. The diversity history is the average positional Hamming distance across pairs of population members; a decrease can indicate that the population's route orderings are becoming more alike. Neither history alone proves global optimality. The parameter experiment later in the notebook is exploratory: it performs one stochastic run per parameter combination, so its results do not estimate run-to-run variability.

## Mathematical formulation and discrete Firefly movement

A route is a permutation of the item set. If a route visits item locations in order, its objective is the total Euclidean length from the depot through the ordered items and back to the depot. The algorithm seeks a route with a small objective value; brightness is defined so that shorter routes are brighter.

Let $V$ be the set of items, let $n = |V|$, let $p_i \in \mathbb{R}^2$ be the coordinate of item $i$, and let $p_0 \in \mathbb{R}^2$ be the depot coordinate. A route $\pi = (\pi_1, \ldots, \pi_n)$ is a permutation of $V$. With Euclidean distance $d(a,b) = \sqrt{(a_x-b_x)^2 + (a_y-b_y)^2}$, its length is

$$L(\pi) = d(p_0,p_{\pi_1}) + \sum_{k=1}^{n-1} d(p_{\pi_k},p_{\pi_{k+1}}) + d(p_{\pi_n},p_0).$$

Here, $L(\pi)$ is the closed-tour distance, and each term measures travel between consecutive stops under the straight-line distance assumption. In the code, `Firefly.calculate_route_distance` computes $L(\pi)$ using the firefly's route, item coordinates, and depot coordinate. Its brightness is

$$I(\pi) = \frac{1}{1 + L(\pi)}.$$

The added 1 keeps the expression defined for a zero-length route, and the reciprocal makes brightness decrease as route length increases. The code implements this relationship in `Firefly.calculate_fitness`.

For two routes $\pi$ and $\sigma$ of the same length, the algorithm measures their positional difference with Hamming distance:

$$r(\pi,\sigma) = \sum_{k=1}^{n} \mathbf{1}[\pi_k \ne \sigma_k],$$

where $\mathbf{1}[\cdot]$ equals 1 when its condition is true and 0 otherwise. This distance measures route-order dissimilarity, not warehouse travel. The attraction coefficient is

$$\beta(r) = \beta_0 e^{-\gamma r^2},$$

where $\beta_0$ is the attraction coefficient at zero route difference and $\gamma$ controls how quickly attraction decreases. A crossover is attempted with probability $\min(1,\beta(r))$, since a probability cannot exceed 1. In the implementation, `calculate_distance_between_fireflies` computes $r$, and `move_firefly_toward` computes the attraction coefficient and invokes `crossover_routes`.

The crossover preserves a segment from the first route and fills unoccupied positions in the order encountered in the target route. The random walk applies either a swap or an inversion mutation. These operators preserve the permutation: each item remains present exactly once.

This is a stochastic search process rather than a dynamic-programming recurrence. Its outer loop is the iteration in `iterate`; within each iteration, the population is considered in pairs. For population size $P$ and route length $n$, pairwise movement and the pairwise diversity calculation each require $O(P^2 n)$ work per iteration. Over $I$ iterations, the overall search therefore has order $O(I P^2 n)$ work, in addition to initialization and plotting. The method does not guarantee that its best route is globally optimal.

In [ ]:
from typing import Dict, List, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fix both random-number generators so that the stochastic search is reproducible.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for the Firefly Algorithm.")

In [ ]:
class Firefly:
    """Represent one candidate picker route and its objective value."""

    def __init__(
        self,
        route: List[int],
        items: Dict[int, Tuple[float, float]],
        depot_location: Tuple[float, float],
    ):
        """Store a route and evaluate its distance and brightness.

        Args:
            route: Ordered item identifiers forming a candidate route.
            items: Mapping from each item identifier to its (x, y) coordinate.
            depot_location: Coordinate of the route's starting and ending depot.
        """
        self.route = route.copy()
        self.items = items
        self.depot_location = depot_location
        self.brightness = 0.0
        self.distance = 0.0
        self.calculate_fitness()

    def calculate_fitness(self) -> None:
        """Update route distance and its inversely related brightness."""
        self.distance = self.calculate_route_distance()
        self.brightness = 1.0 / (1.0 + self.distance)

    def calculate_route_distance(self) -> float:
        """Return the Euclidean distance of the closed route."""
        if not self.route:
            return 0.0

        total_distance = 0.0
        previous_coordinate = self.depot_location

        for item_id in self.route:
            coordinate = self.items[item_id]
            total_distance += np.hypot(
                previous_coordinate[0] - coordinate[0],
                previous_coordinate[1] - coordinate[1],
            )
            previous_coordinate = coordinate

        total_distance += np.hypot(
            previous_coordinate[0] - self.depot_location[0],
            previous_coordinate[1] - self.depot_location[1],
        )
        return float(total_distance)

    def copy(self) -> "Firefly":
        """Return an independently evaluated firefly with the same route."""
        return Firefly(self.route, self.items, self.depot_location)


class FireflyAlgorithmPickerRouting:
    """Search picker routes using discrete firefly attraction and mutation."""

    def __init__(
        self,
        items: Dict[int, Tuple[float, float]],
        depot_location: Tuple[float, float] = (0, 0),
        population_size: int = 30,
        alpha: float = 0.3,
        beta0: float = 1.2,
        gamma: float = 0.02,
    ):
        """Set the instance data and Firefly Algorithm parameters.

        Args:
            items: Mapping from item identifiers to (x, y) coordinates.
            depot_location: Coordinate of the route's starting and ending depot.
            population_size: Number of candidate routes in the population.
            alpha: Probability of applying a random walk, between 0 and 1.
            beta0: Attraction coefficient at zero route difference.
            gamma: Rate at which attraction decreases with route difference.
        """
        if population_size < 1:
            raise ValueError("population_size must be at least 1.")
        if not 0.0 <= alpha <= 1.0:
            raise ValueError("alpha must be between 0 and 1.")
        if beta0 < 0.0 or gamma < 0.0:
            raise ValueError("beta0 and gamma must be non-negative.")
        if len(set(items)) != len(items):
            raise ValueError("Item identifiers must be unique.")

        self.items = items
        self.depot_location = depot_location
        self.population_size = population_size
        self.alpha = alpha
        self.beta0 = beta0
        self.gamma = gamma

        self.population: List[Firefly] = []
        self.best_firefly = None
        self.initial_best_distance = 0.0
        self.convergence_history: List[float] = []
        self.diversity_history: List[float] = []
        self.computation_time = 0.0

    def initialize_population(self) -> None:
        """Create random valid routes and record the initial best route."""
        self.population = []
        item_ids = list(self.items.keys())

        for _ in range(self.population_size):
            random_route = item_ids.copy()
            random.shuffle(random_route)
            self.population.append(
                Firefly(random_route, self.items, self.depot_location)
            )

        # Keep a snapshot because population members are modified during search.
        initial_best = max(self.population, key=lambda firefly: firefly.brightness)
        self.best_firefly = initial_best.copy()
        self.initial_best_distance = self.best_firefly.distance
        self.convergence_history = []
        self.diversity_history = []

        print(f"Initialized a population of {self.population_size} fireflies.")
        print(f"Initial best distance: {self.initial_best_distance:.2f}")

    def calculate_distance_between_fireflies(
        self,
        firefly1: Firefly,
        firefly2: Firefly,
    ) -> float:
        """Return positional Hamming distance between two route orderings."""
        if len(firefly1.route) != len(firefly2.route):
            raise ValueError("Firefly routes must have the same length.")

        distance = sum(
            item1 != item2
            for item1, item2 in zip(firefly1.route, firefly2.route)
        )
        return float(distance)

    def move_firefly_toward(
        self,
        firefly: Firefly,
        target_firefly: Firefly,
        distance: float,
    ) -> None:
        """Apply attraction-based crossover and probabilistic random movement."""
        beta = self.beta0 * np.exp(-self.gamma * distance**2)
        new_route = self.crossover_routes(
            firefly.route,
            target_firefly.route,
            beta,
        )

        if np.random.random() < self.alpha:
            new_route = self.random_walk(new_route)

        firefly.route = new_route
        firefly.calculate_fitness()

    def crossover_routes(
        self,
        route1: List[int],
        route2: List[int],
        beta: float,
    ) -> List[int]:
        """Use order-preserving crossover with attraction-based probability."""
        if len(route1) != len(route2) or set(route1) != set(route2):
            raise ValueError("Crossover routes must contain the same item identifiers.")

        crossover_probability = min(1.0, max(0.0, beta))
        if np.random.random() >= crossover_probability:
            return route1.copy()

        if len(route1) <= 2:
            return route1.copy()

        start = np.random.randint(0, len(route1) - 1)
        end = np.random.randint(start + 1, len(route1))

        child = [None] * len(route1)
        child[start:end] = route1[start:end]
        used_items = set(route1[start:end])
        target_items = [item_id for item_id in route2 if item_id not in used_items]
        target_index = 0

        for index in range(len(child)):
            if child[index] is None:
                child[index] = target_items[target_index]
                target_index += 1

        return child

    def random_walk(self, route: List[int]) -> List[int]:
        """Apply a random swap or inversion while preserving the permutation."""
        if len(route) <= 2:
            return route.copy()

        new_route = route.copy()

        if np.random.random() < 0.5:
            first, second = np.random.choice(len(route), 2, replace=False)
            new_route[first], new_route[second] = (
                new_route[second],
                new_route[first],
            )
        elif np.random.random() < 0.5:
            first, second = np.random.choice(len(route), 2, replace=False)
            if first > second:
                first, second = second, first
            new_route[first : second + 1] = new_route[first : second + 1][::-1]

        return new_route

    def calculate_population_diversity(self) -> float:
        """Return the mean pairwise positional Hamming distance."""
        if len(self.population) <= 1:
            return 0.0

        total_distance = 0.0
        pair_count = 0

        for first_index in range(len(self.population)):
            for second_index in range(first_index + 1, len(self.population)):
                total_distance += self.calculate_distance_between_fireflies(
                    self.population[first_index],
                    self.population[second_index],
                )
                pair_count += 1

        return total_distance / pair_count

    def iterate(
        self,
        max_iterations: int = 1000,
        convergence_threshold: float = 1e-6,
    ) -> Dict[str, object]:
        """Run the search and return the best route and recorded histories.

        The stopping check compares the newest best distance with the value
        ten history entries earlier, after at least eleven entries exist.
        """
        if not self.population or self.best_firefly is None:
            raise RuntimeError("Call initialize_population before iterate.")
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1.")
        if convergence_threshold < 0.0:
            raise ValueError("convergence_threshold must be non-negative.")

        start_time = time.time()
        print(f"Starting the Firefly Algorithm for up to {max_iterations} iterations.")

        for iteration in range(max_iterations):
            self.population.sort(
                key=lambda firefly: firefly.brightness,
                reverse=True,
            )

            for first_index in range(len(self.population)):
                for second_index in range(first_index + 1, len(self.population)):
                    first_firefly = self.population[first_index]
                    second_firefly = self.population[second_index]
                    distance = self.calculate_distance_between_fireflies(
                        first_firefly,
                        second_firefly,
                    )

                    # Compare current brightness because movement can change rank.
                    if first_firefly.brightness > second_firefly.brightness:
                        self.move_firefly_toward(
                            second_firefly,
                            first_firefly,
                            distance,
                        )
                    elif second_firefly.brightness > first_firefly.brightness:
                        self.move_firefly_toward(
                            first_firefly,
                            second_firefly,
                            distance,
                        )

            current_best = max(
                self.population,
                key=lambda firefly: firefly.brightness,
            )
            if current_best.brightness > self.best_firefly.brightness:
                self.best_firefly = current_best.copy()

            self.convergence_history.append(self.best_firefly.distance)
            self.diversity_history.append(self.calculate_population_diversity())

            if iteration % 100 == 0:
                print(
                    f"Iteration {iteration + 1}: "
                    f"best distance = {self.best_firefly.distance:.2f}, "
                    f"diversity = {self.diversity_history[-1]:.2f}"
                )

            if len(self.convergence_history) > 10:
                recent_improvement = abs(
                    self.convergence_history[-1]
                    - self.convergence_history[-10]
                )
                if recent_improvement < convergence_threshold:
                    print(f"Convergence criterion met at iteration {iteration + 1}.")
                    break

        self.computation_time = time.time() - start_time
        iterations_completed = len(self.convergence_history)

        print(f"Search completed after {iterations_completed} iterations.")
        print(f"Best distance found: {self.best_firefly.distance:.2f}")
        print(f"Computation time: {self.computation_time:.3f} seconds")

        return {
            "best_route": self.best_firefly.route.copy(),
            "best_distance": self.best_firefly.distance,
            "initial_best_distance": self.initial_best_distance,
            "iterations": iterations_completed,
            "computation_time": self.computation_time,
            "convergence_history": self.convergence_history.copy(),
            "diversity_history": self.diversity_history.copy(),
        }


print("FireflyAlgorithmPickerRouting class is ready.")

In [ ]:
# Define the 10-item example and its depot using the stated coordinates.
items = {
    1: (2, 3),
    2: (5, 7),
    3: (8, 2),
    4: (12, 6),
    5: (3, 8),
    6: (9, 4),
    7: (14, 1),
    8: (6, 5),
    9: (11, 7),
    10: (4, 2),
}

depot_location = (0, 0)

print("=== Picker Routing Problem: Firefly Algorithm ===")
print(f"Depot location: {depot_location}")
print("Items to collect:")
for item_id, coordinate in items.items():
    print(f"  Item {item_id}: {coordinate}")

firefly_solver = FireflyAlgorithmPickerRouting(
    items=items,
    depot_location=depot_location,
    population_size=30,
    alpha=0.3,
    beta0=1.2,
    gamma=0.02,
)

firefly_solver.initialize_population()
solution = firefly_solver.iterate(max_iterations=1000)

## Inspect the best route

The next cell reports the best candidate found by the stochastic search. Its improvement is measured against the best member of the randomly initialized population. The detailed route table accumulates each Euclidean leg, including the final return to the depot, so its last cumulative distance should agree with the reported route distance.

In [ ]:
best_route = solution["best_route"]
initial_distance = solution["initial_best_distance"]
final_distance = solution["best_distance"]

assert len(best_route) == len(items)
assert set(best_route) == set(items)
assert np.isclose(
    Firefly(best_route, items, depot_location).distance,
    final_distance,
)
assert len(solution["convergence_history"]) == solution["iterations"]
assert len(solution["diversity_history"]) == solution["iterations"]

improvement = (
    (initial_distance - final_distance) / initial_distance * 100.0
    if initial_distance > 0.0
    else 0.0
)

print("=== Solution Analysis ===")
print(f"Best route: {best_route}")
print(f"Best distance: {final_distance:.2f} distance units")
print(f"Iterations completed: {solution['iterations']}")
print(f"Computation time: {solution['computation_time']:.3f} seconds")

print("\n=== Search Improvement ===")
print(f"Initial population's best distance: {initial_distance:.2f}")
print(f"Best distance found: {final_distance:.2f}")
print(f"Improvement from initial best: {improvement:.1f}%")

print("\n=== Detailed Route Analysis ===")
print("Step | Stop  | Coordinates | Cumulative distance")
print("-" * 56)

cumulative_distance = 0.0
current_position = depot_location

for step, item_id in enumerate(best_route, start=1):
    item_coordinate = items[item_id]
    leg_distance = np.hypot(
        current_position[0] - item_coordinate[0],
        current_position[1] - item_coordinate[1],
    )
    cumulative_distance += leg_distance
    print(
        f"{step:4d} | Item {item_id:<2d} | {str(item_coordinate):<11s} | "
        f"{cumulative_distance:8.2f}"
    )
    current_position = item_coordinate

return_distance = np.hypot(
    current_position[0] - depot_location[0],
    current_position[1] - depot_location[1],
)
cumulative_distance += return_distance
print(
    f"{len(best_route) + 1:4d} | Depot   | {str(depot_location):<11s} | "
    f"{cumulative_distance:8.2f}"
)
assert np.isclose(cumulative_distance, final_distance)

## Visualize convergence, route, and diversity

The convergence plot includes the initial population's best distance as its first point, followed by the best distance recorded after each iteration. The route plot displays the best closed tour found. The diversity plot uses positional Hamming distance, so its vertical axis describes differences in item ordering rather than physical distance.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Picker Routing: Firefly Algorithm Results",
    fontsize=16,
    fontweight="bold",
)

# Include the initialization result so the plot shows improvement from generation zero.
convergence_values = [initial_distance] + solution["convergence_history"]
convergence_iterations = range(len(convergence_values))
ax1 = axes[0, 0]
ax1.plot(
    convergence_iterations,
    convergence_values,
    color="blue",
    linewidth=2,
    label="Best distance found",
)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Route distance (coordinate distance units)")
ax1.set_title("Best-Route Convergence")
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.annotate(
    f"Improvement: {improvement:.1f}%",
    xy=(len(convergence_values) - 1, final_distance),
    xytext=(0.55, 0.9),
    textcoords="axes fraction",
    ha="center",
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "lightyellow", "alpha": 0.8},
)

ax2 = axes[0, 1]
ax2.plot(
    depot_location[0],
    depot_location[1],
    marker="s",
    color="black",
    markersize=10,
    linestyle="None",
    label="Depot",
)
for item_id, coordinate in items.items():
    ax2.plot(
        coordinate[0],
        coordinate[1],
        marker="o",
        color="tab:red",
        markersize=7,
        linestyle="None",
        label="Item locations" if item_id == next(iter(items)) else None,
    )
    ax2.annotate(
        str(item_id),
        coordinate,
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=9,
        fontweight="bold",
    )

route_coordinates = [depot_location] + [items[item_id] for item_id in best_route]
route_coordinates.append(depot_location)
route_x = [coordinate[0] for coordinate in route_coordinates]
route_y = [coordinate[1] for coordinate in route_coordinates]
ax2.plot(
    route_x,
    route_y,
    color="tab:blue",
    linewidth=1.8,
    alpha=0.8,
    label="Route",
)
ax2.set_xlabel("X coordinate (coordinate units)")
ax2.set_ylabel("Y coordinate (coordinate units)")
ax2.set_title(f"Best Route Found (distance: {final_distance:.2f})")
ax2.grid(True, alpha=0.3)
ax2.set_aspect("equal", adjustable="datalim")
ax2.legend()

ax3 = axes[1, 0]
diversity_iterations = range(1, len(solution["diversity_history"]) + 1)
ax3.plot(
    diversity_iterations,
    solution["diversity_history"],
    color="tab:green",
    linewidth=2,
    label="Mean pairwise Hamming distance",
)
ax3.set_xlabel("Iteration")
ax3.set_ylabel("Mean positional Hamming distance")
ax3.set_title("Population Route-Ordering Diversity")
ax3.grid(True, alpha=0.3)
ax3.legend()

ax4 = axes[1, 1]
leg_distances = []
for start_coordinate, end_coordinate in zip(
    route_coordinates[:-1],
    route_coordinates[1:],
):
    leg_distances.append(
        float(
            np.hypot(
                start_coordinate[0] - end_coordinate[0],
                start_coordinate[1] - end_coordinate[1],
            )
        )
    )
leg_labels = [f"{index + 1}: {best_route[index]}" for index in range(len(best_route))]
leg_labels.append("Return to depot")
ax4.bar(
    range(1, len(leg_distances) + 1),
    leg_distances,
    color="tab:orange",
    alpha=0.8,
    label="Leg distance",
)
ax4.set_xlabel("Route leg (item identifier shown for item stops)")
ax4.set_ylabel("Leg distance (coordinate distance units)")
ax4.set_title("Distance by Route Leg")
ax4.set_xticks(range(1, len(leg_labels) + 1))
ax4.set_xticklabels(leg_labels, rotation=45, ha="right")
ax4.grid(True, axis="y", alpha=0.3)
ax4.legend()

plt.tight_layout()
plt.show()

print("The plots show the search history, best route, route-ordering diversity, and leg distances.")

## Parameter sensitivity experiment

The following experiment varies the random-walk probability `alpha`, the initial attraction coefficient `beta0`, and the light-absorption coefficient `gamma`. Each combination uses the same item instance, a population of 20, and a maximum of 500 iterations, as specified for this experiment. The global random-number generators continue from the fixed seeds set near the beginning of the notebook; each parameter combination receives one run, not a repeated-sample estimate.

The plots summarize the observed best distances by parameter value and show the relationship between stopping iteration, route distance, and elapsed time. These results describe this particular seeded experiment. They should not be interpreted as proof that a parameter setting is generally best or that the corresponding route is globally optimal.

In [ ]:
def test_parameter_combinations() -> List[Dict[str, float]]:
    """Run one seeded-stream search for each parameter combination."""
    alphas = [0.1, 0.3, 0.5]
    beta0_values = [0.8, 1.2, 1.6]
    gamma_values = [0.01, 0.02, 0.04]
    results = []

    for alpha in alphas:
        for beta0 in beta0_values:
            for gamma in gamma_values:
                print(
                    f"\nTesting alpha={alpha}, beta0={beta0}, gamma={gamma}"
                )
                test_solver = FireflyAlgorithmPickerRouting(
                    items=items,
                    depot_location=depot_location,
                    population_size=20,
                    alpha=alpha,
                    beta0=beta0,
                    gamma=gamma,
                )
                test_solver.initialize_population()
                test_solution = test_solver.iterate(max_iterations=500)

                results.append(
                    {
                        "alpha": alpha,
                        "beta0": beta0,
                        "gamma": gamma,
                        "best_distance": test_solution["best_distance"],
                        "iterations": test_solution["iterations"],
                        "time": test_solution["computation_time"],
                    }
                )
                print(f"Best distance: {test_solution['best_distance']:.2f}")
                print(f"Iterations completed: {test_solution['iterations']}")

    return results


print("=== Parameter Sensitivity Analysis ===")
sensitivity_results = test_parameter_combinations()
df = pd.DataFrame(sensitivity_results)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(
    "Firefly Algorithm: Parameter Sensitivity",
    fontsize=16,
    fontweight="bold",
)

ax1 = axes[0, 0]
alpha_groups = df.groupby("alpha")["best_distance"].mean()
ax1.bar(alpha_groups.index, alpha_groups.values, color="skyblue", alpha=0.8)
ax1.set_xlabel("Alpha (random-walk probability)")
ax1.set_ylabel("Mean best route distance (coordinate distance units)")
ax1.set_title("Observed Distance by Alpha")
ax1.grid(True, axis="y", alpha=0.3)

ax2 = axes[0, 1]
beta0_groups = df.groupby("beta0")["best_distance"].mean()
ax2.bar(beta0_groups.index, beta0_groups.values, color="lightcoral", alpha=0.8)
ax2.set_xlabel("Beta0 (attraction coefficient at zero distance)")
ax2.set_ylabel("Mean best route distance (coordinate distance units)")
ax2.set_title("Observed Distance by Beta0")
ax2.grid(True, axis="y", alpha=0.3)

ax3 = axes[1, 0]
gamma_groups = df.groupby("gamma")["best_distance"].mean()
ax3.bar(gamma_groups.index, gamma_groups.values, color="lightgreen", alpha=0.8)
ax3.set_xlabel("Gamma (attraction decay coefficient)")
ax3.set_ylabel("Mean best route distance (coordinate distance units)")
ax3.set_title("Observed Distance by Gamma")
ax3.grid(True, axis="y", alpha=0.3)

ax4 = axes[1, 1]
scatter = ax4.scatter(
    df["iterations"],
    df["best_distance"],
    c=df["time"],
    s=80,
    alpha=0.75,
    cmap="viridis",
)
ax4.set_xlabel("Iterations completed")
ax4.set_ylabel("Best route distance (coordinate distance units)")
ax4.set_title("Stopping Iteration and Route Distance")
ax4.grid(True, alpha=0.3)
colorbar = plt.colorbar(scatter, ax=ax4)
colorbar.set_label("Computation time (seconds)")

plt.tight_layout()
plt.show()

best_result = min(sensitivity_results, key=lambda result: result["best_distance"])
print("\n=== Best Observed Parameter Combination in This Experiment ===")
print(f"Alpha: {best_result['alpha']}")
print(f"Beta0: {best_result['beta0']}")
print(f"Gamma: {best_result['gamma']}")
print(f"Best distance: {best_result['best_distance']:.2f}")
print(f"Iterations completed: {best_result['iterations']}")
print(f"Computation time: {best_result['time']:.3f} seconds")

print("\n=== Mean Results Grouped by Parameter Value ===")
print("Parameter value | Mean best distance | Mean iterations | Mean time (seconds)")
print("-" * 76)

for parameter in ["alpha", "beta0", "gamma"]:
    for value, group in df.groupby(parameter):
        print(
            f"{parameter}={value:<5} | "
            f"{group['best_distance'].mean():17.2f} | "
            f"{group['iterations'].mean():15.1f} | "
            f"{group['time'].mean():19.3f}"
        )

## Interpreting this tier

The Firefly Algorithm represents each candidate as a complete route and searches a population of such routes. In this discrete adaptation, brighter routes attract dimmer routes through order-preserving crossover, while swap and inversion mutations provide random exploration. Route quality is measured by the closed-tour Euclidean distance, whereas attraction and population diversity use positional Hamming distance between permutations.

The method is useful as a flexible stochastic search when a high-quality route is sought and an optimality certificate is not required. Its outcome depends on the initial population, random choices, and parameter settings. A fixed seed makes this notebook's sequence of random choices reproducible in a compatible software environment, but elapsed time can vary across machines. The single-run sensitivity experiment is illustrative rather than a statistical comparison.

The model's conclusions apply to the stated coordinate and straight-line-distance assumptions. Real warehouse routing may require a network-distance model that represents aisles and cross-aisles, as well as operational constraints such as blocked paths or picking policies. Those considerations are outside the scope of this instance.